# Все активные кассы: старая модель, центральный SARIMA и closed_nan q01..q15

Самостоятельный ретро-расчёт NS и центрального `saldo_turn`. Для каждой даты скоринга используется 12 месяцев истории до `T−2`, строится 31 модельный шаг и сохраняются 30 дат горизонта. Обычные пропуски в обучении остаются `NaN`, наблюдаемые нули сохраняются, исключённый период также остаётся `NaN`.

Расчёт охватывает динамический набор всех активных касс, поддерживает атомарные checkpoints по датам и не пишет результаты в Oracle.

In [ ]:
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from joblib import Parallel, delayed, parallel_backend
from prefect.blocks.system import Secret
from sqlalchemy import text
from statsmodels.tsa.statespace.sarimax import SARIMAX
from toolbox import oracle

## 1. Параметры

Границы ретро, лаг и горизонт задаются параметрами; число дат нигде не фиксируется вручную. Настройки guarded SARIMA и исключённый диапазон совпадают с проверенным ретро-notebook.

In [ ]:
SOURCE_TABLE = "AIDA2.AIDA_TRS_DTM_CASHOP@aida"
OLD_FORECAST_TABLE = "AIDA2.AIDA_TRS_DTM_FORECAST_HISTORY@aida"

RETRO_DATE_FROM = pd.Timestamp("2026-04-15")
RETRO_DATE_TO = pd.Timestamp("2026-06-15")
HISTORY_MONTHS = 12
ACTIVE_LOOKBACK_MONTHS = 1
DATA_LAG_DAYS = 2
FORECAST_DAYS = 30
MODEL_STEPS = FORECAST_DAYS + DATA_LAG_DAYS - 1

INITIAL_TRAIN_DAYS = 60
BACKTEST_STEP_DAYS = 7
BOOTSTRAP_ITERATIONS = 1000
BOOTSTRAP_QUANTILES = np.round(np.arange(0.01, 0.16, 0.01), 2)
MIN_BACKTEST_ERROR_BLOCKS = 5
MIN_SARIMA_DAYS = 90
SARIMA_NO_ERROR_HISTORY_ADJUSTMENT = 0.15
SARIMA_ORDER = (1, 1, 1)
SARIMA_SEASONAL_ORDER = (1, 0, 1, 7)
CASH_NEED_CLIP_UPPER = 0.0
FORECAST_SCALE_MULTIPLIER = 20.0
EXCLUDE_DATE_RANGES = [("2025-09-01", "2025-11-01")]

RANDOM_SEED = 42
N_JOBS = 16
LOAD_RAW_FROM_CACHE = False
USE_CHECKPOINTS = True
RAW_CACHE_PATH = Path("data/raw/cashdesk_all_cash_closed_nan_saldo_v2_raw.parquet")
OLD_CACHE_PATH = Path("data/raw/cashdesk_all_cash_closed_nan_saldo_v2_old.parquet")
OUTPUT_DIR = Path("outputs/all_cash_closed_nan_saldo_v2")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

score_dates = pd.date_range(RETRO_DATE_FROM, RETRO_DATE_TO, freq="D")
MIN_REPORT_DATE = score_dates.min() - pd.Timedelta(days=DATA_LAG_DAYS)
DATA_DATE_FROM = MIN_REPORT_DATE - pd.DateOffset(months=HISTORY_MONTHS)
FACT_DATE_TO_EXCLUSIVE = score_dates.max() + pd.Timedelta(days=FORECAST_DAYS)
OLD_DATE_TO_EXCLUSIVE = RETRO_DATE_TO + pd.Timedelta(days=1)
QUANTILE_COLUMNS = {
    float(quantile): "atdtmco_ns_pred_q{:02d}".format(int(round(100 * quantile)))
    for quantile in BOOTSTRAP_QUANTILES
}
PREDICTION_COLUMNS = [
    "atdtmco_saldo_turn_pred",
    "atdtmco_ns_pred_central",
] + list(QUANTILE_COLUMNS.values())
CHECKPOINT_COLUMNS = [
    "score_date",
    "report_date",
    "atdtmco_cashdesk_name",
    "atdtmco_cashdesk_name_trn",
    "forecast_date",
] + PREDICTION_COLUMNS + ["fallback saldo", "fallback NS", "bootstrap NS"]

print("Дат скоринга: {}".format(len(score_dates)))
print("История/факты: {} — {}".format(
    DATA_DATE_FROM.date(),
    (FACT_DATE_TO_EXCLUSIVE - pd.Timedelta(days=1)).date(),
))

## 2. Однократная загрузка и подготовка

Oracle либо parquet-кэши читаются один раз в parent-процессе. `forecast_model` хранится положительным и переводится на общую отрицательную шкалу. Факты остаются исходными: пропуски не заменяются нулями.

In [ ]:
USERNAME_CDW = "sb_analytics"
engine_cdw = None


async def create_cdw_engine():
    password_cdw = (await Secret.load("pass-sb-analytics")).get()
    return oracle.create_engine_cdw(USERNAME_CDW, password_cdw)


if LOAD_RAW_FROM_CACHE:
    raw_df = pd.read_parquet(RAW_CACHE_PATH)
    old_history_df = pd.read_parquet(OLD_CACHE_PATH)
else:
    engine_cdw = await create_cdw_engine()
    source_query = """
    select
        atdtmco_cashdesk_name,
        atdtmco_cashdesk_name_trn,
        atdtmco_calday,
        atdtmco_saldo_turn,
        atdtmco_ns
    from {source_table}
    where atdtmco_calday >= date '{date_from}'
      and atdtmco_calday < date '{date_to}'
    """.format(
        source_table=SOURCE_TABLE,
        date_from=DATA_DATE_FROM.date().isoformat(),
        date_to=FACT_DATE_TO_EXCLUSIVE.date().isoformat(),
    )
    for exclude_start, exclude_end in EXCLUDE_DATE_RANGES:
        source_query += (
            "\n  and not (atdtmco_calday >= date '{}' and atdtmco_calday < date '{}')"
            .format(exclude_start, exclude_end)
        )
    old_query = """
    select cashdesk_name, calday, flow_minimum, forecast_model, forecast_time
    from {old_table}
    where calday >= date '{date_from}'
      and calday < date '{date_to}'
    """.format(
        old_table=OLD_FORECAST_TABLE,
        date_from=RETRO_DATE_FROM.date().isoformat(),
        date_to=OLD_DATE_TO_EXCLUSIVE.date().isoformat(),
    )
    with engine_cdw.connect() as conn:
        raw_df = pd.read_sql(text(source_query), conn)
        old_history_df = pd.read_sql(text(old_query), conn)

    RAW_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    raw_df.to_parquet(RAW_CACHE_PATH, index=False)
    old_history_df.to_parquet(OLD_CACHE_PATH, index=False)

raw_df.columns = raw_df.columns.str.lower()
old_history_df.columns = old_history_df.columns.str.lower()
raw_df["atdtmco_calday"] = pd.to_datetime(raw_df["atdtmco_calday"]).dt.normalize()
old_history_df["calday"] = pd.to_datetime(old_history_df["calday"]).dt.normalize()
old_history_df["forecast_time"] = pd.to_datetime(
    old_history_df["forecast_time"], errors="coerce"
)

daily_df = (
    raw_df
    .sort_values(["atdtmco_cashdesk_name", "atdtmco_calday"])
    .groupby(
        ["atdtmco_cashdesk_name", "atdtmco_calday"],
        as_index=False,
        dropna=False,
    )
    .agg(
        atdtmco_cashdesk_name_trn=("atdtmco_cashdesk_name_trn", "last"),
        atdtmco_saldo_turn_fact=("atdtmco_saldo_turn", "sum"),
        atdtmco_ns_daily_min_raw=("atdtmco_ns", "min"),
    )
    .rename(columns={"atdtmco_calday": "calday"})
    .sort_values(["atdtmco_cashdesk_name", "calday"])
    .reset_index(drop=True)
)
daily_df["atdtmco_ns_fact"] = pd.to_numeric(
    daily_df["atdtmco_ns_daily_min_raw"], errors="coerce"
).clip(upper=CASH_NEED_CLIP_UPPER)

old_history_dedup_df = (
    old_history_df
    .sort_values("forecast_time")
    .drop_duplicates(["cashdesk_name", "calday"], keep="last")
    .reset_index(drop=True)
)
old_history_dedup_df["atdtmco_ns_pred_old"] = -pd.to_numeric(
    old_history_dedup_df["forecast_model"], errors="coerce"
)
print("Загружено {:,} дневных строк и {:,} старых прогнозов".format(
    len(daily_df), len(old_history_dedup_df)
))

## 3. Модельное ядро

Guarded SARIMA отклоняет несошедшиеся, неустойчивые, нечисловые и несоразмерные модели. Финальная SARIMA NS обучается один раз на задачу: центральный прогноз не содержит поправок на исторические ошибки, но соблюдает физическое ограничение `NS ≤ 0`; те же центральные значения переиспользуются как база q-сетки. Скользящие ретро-проверки собирают конечные ошибки отдельно по каждому шагу; при числе ошибок меньше порога только этот шаг получает поправку 15%.

Для `saldo_turn` строится отдельный центральный guarded SARIMA без bootstrap. Пропущенные дни истории остаются `NaN`, наблюдаемые нули сохраняются; при короткой истории или отказе модели применяется резервный прогноз по дням недели.

In [ ]:
@dataclass(frozen=True)
class SarimaConfig:
    order: Tuple[int, int, int]
    seasonal_order: Tuple[int, int, int, int]
    maxiter: int = 200


SARIMA_CONFIG = SarimaConfig(
    order=SARIMA_ORDER,
    seasonal_order=SARIMA_SEASONAL_ORDER,
)


def make_closed_nan_series(
    cashdesk_df: pd.DataFrame,
    value_col: str,
    report_date: pd.Timestamp,
) -> pd.Series:
    observed = (
        cashdesk_df.set_index("calday")[value_col]
        .sort_index()
        .astype(float)
    )
    if observed.empty:
        raise ValueError("Нет наблюдаемой истории {}".format(value_col))
    history_date_from = report_date - pd.DateOffset(months=HISTORY_MONTHS)
    full_index = pd.date_range(history_date_from, report_date, freq="D")
    series = observed.reindex(full_index)
    for exclude_start, exclude_end in EXCLUDE_DATE_RANGES:
        excluded_mask = (
            (series.index >= pd.Timestamp(exclude_start))
            & (series.index < pd.Timestamp(exclude_end))
        )
        series.loc[excluded_mask] = np.nan
    series.index.name = "calday"
    return series


def make_future_index(y: pd.Series, steps: int) -> pd.DatetimeIndex:
    return pd.date_range(
        y.index.max() + pd.Timedelta(days=1), periods=steps, freq="D"
    )


def guarded_sarima_forecast(
    y: pd.Series,
    steps: int,
    config: SarimaConfig,
) -> Tuple[object, np.ndarray]:
    model = SARIMAX(
        y,
        order=config.order,
        seasonal_order=config.seasonal_order,
        enforce_stationarity=True,
        enforce_invertibility=True,
    )
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Non-invertible starting MA parameters found.*",
            category=UserWarning,
        )
        warnings.filterwarnings(
            "ignore",
            message="Non-stationary starting autoregressive parameters found.*",
            category=UserWarning,
        )
        warnings.filterwarnings(
            "ignore",
            message="Non-stationary starting seasonal autoregressive parameters found.*",
            category=UserWarning,
        )
        warnings.filterwarnings(
            "ignore",
            message="Non-invertible starting seasonal moving average parameters found.*",
            category=UserWarning,
        )
        warnings.filterwarnings(
            "ignore", message="Maximum Likelihood optimization failed to converge.*"
        )
        fitted = model.fit(disp=False, maxiter=config.maxiter)
    if not bool(fitted.mle_retvals.get("converged", False)):
        raise ValueError("SARIMA не сошлась")
    for root_name, roots in (("AR", fitted.arroots), ("MA", fitted.maroots)):
        root_modulus = np.abs(np.asarray(roots, dtype=complex))
        if root_modulus.size and (
            not np.isfinite(root_modulus).all() or (root_modulus <= 1.0).any()
        ):
            raise ValueError("Неустойчивые {}-корни".format(root_name))
    forecast = fitted.get_forecast(steps=steps).predicted_mean.to_numpy(dtype=float)
    valid_history = y.dropna().to_numpy(dtype=float)
    history_scale = max(1.0, float(np.quantile(np.abs(valid_history), 0.99)))
    if not np.isfinite(forecast).all():
        raise ValueError("Нечисловой SARIMA-прогноз")
    if (np.abs(forecast) > FORECAST_SCALE_MULTIPLIER * history_scale).any():
        raise ValueError("Несоразмерный SARIMA-прогноз")
    return fitted, forecast


def weekday_point_forecast(
    y: pd.Series,
    steps: int,
    clip_upper_zero: bool,
) -> np.ndarray:
    y_clean = y.dropna()
    global_value = float(y_clean.median())
    weekday_values = y_clean.groupby(y_clean.index.dayofweek).median()
    values = np.asarray([
        weekday_values.get(day.dayofweek, global_value)
        for day in make_future_index(y, steps)
    ], dtype=float)
    if clip_upper_zero:
        return np.minimum(values, CASH_NEED_CLIP_UPPER)
    return values


def weekday_quantile_values(
    y: pd.Series,
    steps: int,
    quantile: float,
) -> np.ndarray:
    y_clean = y.dropna()
    global_quantile = float(y_clean.quantile(quantile))
    weekday_quantiles = y_clean.groupby(y_clean.index.dayofweek).quantile(quantile)
    values = np.asarray([
        weekday_quantiles.get(day.dayofweek, global_quantile)
        for day in make_future_index(y, steps)
    ], dtype=float)
    return np.minimum(values, CASH_NEED_CLIP_UPPER)


def collect_closed_nan_error_blocks(
    y: pd.Series,
    steps: int,
    config: SarimaConfig,
) -> np.ndarray:
    error_blocks = []
    max_train_end = len(y) - steps
    for train_end in range(
        INITIAL_TRAIN_DAYS,
        max_train_end + 1,
        BACKTEST_STEP_DAYS,
    ):
        train = y.iloc[:train_end]
        test = y.iloc[train_end:train_end + steps]
        if train.notna().sum() < INITIAL_TRAIN_DAYS:
            continue
        try:
            _, forecast = guarded_sarima_forecast(train, steps, config)
        except Exception:
            continue
        error = test.to_numpy(dtype=float) - forecast
        if np.isfinite(error).any():
            error_blocks.append(error)
    if not error_blocks:
        return np.empty((0, steps), dtype=float)
    return np.vstack(error_blocks)


def build_per_step_quantile_grid(
    central_values: np.ndarray,
    error_blocks: np.ndarray,
    random_generator: np.random.Generator,
) -> Tuple[Dict[str, np.ndarray], np.ndarray]:
    adjusted = np.minimum(
        central_values * (1.0 + SARIMA_NO_ERROR_HISTORY_ADJUSTMENT),
        CASH_NEED_CLIP_UPPER,
    )
    finite_residual_counts = np.isfinite(error_blocks).sum(axis=0)
    bootstrap_step_mask = finite_residual_counts >= MIN_BACKTEST_ERROR_BLOCKS
    quantile_grid = {
        QUANTILE_COLUMNS[float(quantile)]: adjusted.copy()
        for quantile in BOOTSTRAP_QUANTILES
    }
    for step_index in np.flatnonzero(bootstrap_step_mask):
        finite_residuals = error_blocks[:, step_index]
        finite_residuals = finite_residuals[np.isfinite(finite_residuals)]
        sampled_residuals = random_generator.choice(
            finite_residuals,
            size=BOOTSTRAP_ITERATIONS,
            replace=True,
        )
        step_scenarios = np.minimum(
            central_values[step_index] + sampled_residuals,
            CASH_NEED_CLIP_UPPER,
        )
        step_quantiles = np.quantile(step_scenarios, BOOTSTRAP_QUANTILES)
        for quantile_index, quantile in enumerate(BOOTSTRAP_QUANTILES):
            quantile_grid[QUANTILE_COLUMNS[float(quantile)]][step_index] = (
                step_quantiles[quantile_index]
            )
    return quantile_grid, bootstrap_step_mask


def forecast_closed_nan_quantile_grid(
    y: pd.Series,
    steps: int,
    random_generator: np.random.Generator,
) -> Tuple[np.ndarray, Dict[str, np.ndarray], np.ndarray]:
    error_blocks = collect_closed_nan_error_blocks(y, steps, SARIMA_CONFIG)
    _, sarima_mean = guarded_sarima_forecast(y, steps, SARIMA_CONFIG)
    central_values = np.minimum(sarima_mean, CASH_NEED_CLIP_UPPER)
    quantile_grid, bootstrap_step_mask = build_per_step_quantile_grid(
        central_values,
        error_blocks,
        random_generator,
    )
    return central_values, quantile_grid, bootstrap_step_mask

## 4. Параллельный ретро-прогон и checkpoints

Стабильный индекс RNG берётся из исходного отсортированного списка активных касс. Один пул `loky` используется для всех дат. Каждый checkpoint проверяется против текущего динамического набора касс и записывается атомарно только после проверки полной даты.

In [ ]:
def forecast_cashdesk_task(
    score_date: pd.Timestamp,
    report_date: pd.Timestamp,
    cashdesk_index: int,
    cashdesk_name: str,
    cashdesk_df: pd.DataFrame,
) -> pd.DataFrame:
    history_df = cashdesk_df.sort_values("calday").copy()
    ns_series = make_closed_nan_series(
        history_df,
        "atdtmco_ns_fact",
        report_date,
    )
    saldo_series = make_closed_nan_series(
        history_df,
        "atdtmco_saldo_turn_fact",
        report_date,
    )
    future_index = make_future_index(ns_series, MODEL_STEPS)
    random_generator = np.random.default_rng(np.random.SeedSequence([
        RANDOM_SEED,
        int(score_date.toordinal()),
        int(cashdesk_index),
    ]))
    fallback_saldo = False
    try:
        if saldo_series.notna().sum() < MIN_SARIMA_DAYS:
            raise ValueError("Короткая история saldo")
        _, saldo_values = guarded_sarima_forecast(
            saldo_series,
            MODEL_STEPS,
            SARIMA_CONFIG,
        )
    except Exception:
        fallback_saldo = True
        saldo_values = weekday_point_forecast(
            saldo_series,
            MODEL_STEPS,
            False,
        )
    fallback_ns = False
    bootstrap_step_mask = np.zeros(MODEL_STEPS, dtype=bool)
    try:
        if ns_series.notna().sum() < MIN_SARIMA_DAYS:
            raise ValueError("Короткая история NS")
        central_values, quantile_grid, bootstrap_step_mask = (
            forecast_closed_nan_quantile_grid(
                ns_series,
                MODEL_STEPS,
                random_generator,
            )
        )
    except Exception:
        fallback_ns = True
        central_values = weekday_point_forecast(
            ns_series,
            MODEL_STEPS,
            True,
        )
        quantile_grid = {
            QUANTILE_COLUMNS[float(quantile)]: weekday_quantile_values(
                ns_series,
                MODEL_STEPS,
                float(quantile),
            )
            for quantile in BOOTSTRAP_QUANTILES
        }
    translated_names = history_df["atdtmco_cashdesk_name_trn"].dropna()
    translated_name = translated_names.iloc[-1] if len(translated_names) else pd.NA
    result_df = pd.DataFrame({
        "forecast_date": future_index,
        "atdtmco_saldo_turn_pred": saldo_values,
        "atdtmco_ns_pred_central": central_values,
        "bootstrap NS": bootstrap_step_mask,
    })
    for column_name, values in quantile_grid.items():
        result_df[column_name] = values
    result_df = result_df[
        (result_df["forecast_date"] >= score_date)
        & (result_df["forecast_date"] < score_date + pd.Timedelta(days=FORECAST_DAYS))
    ].copy()
    result_df.insert(0, "score_date", score_date)
    result_df.insert(1, "report_date", report_date)
    result_df.insert(2, "atdtmco_cashdesk_name", cashdesk_name)
    result_df.insert(3, "atdtmco_cashdesk_name_trn", translated_name)
    result_df["fallback saldo"] = bool(fallback_saldo)
    result_df["fallback NS"] = bool(fallback_ns)
    return result_df[CHECKPOINT_COLUMNS]


def build_scoring_tasks(
    score_date: pd.Timestamp,
    all_daily_df: pd.DataFrame,
) -> Tuple[pd.Timestamp, List[Tuple[int, str, pd.DataFrame]]]:
    report_date = score_date - pd.Timedelta(days=DATA_LAG_DAYS)
    history_date_from = report_date - pd.DateOffset(months=HISTORY_MONTHS)
    active_date_from = report_date - pd.DateOffset(months=ACTIVE_LOOKBACK_MONTHS)
    available_df = all_daily_df[
        (all_daily_df["calday"] >= history_date_from)
        & (all_daily_df["calday"] <= report_date)
    ]
    active_cashdesks = sorted(
        available_df.loc[
            available_df["calday"] >= active_date_from,
            "atdtmco_cashdesk_name",
        ].dropna().unique().tolist()
    )
    tasks = []
    for cashdesk_index, cashdesk_name in enumerate(active_cashdesks):
        cashdesk_df = available_df[
            available_df["atdtmco_cashdesk_name"] == cashdesk_name
        ].copy()
        tasks.append((cashdesk_index, cashdesk_name, cashdesk_df))
    return report_date, tasks


def validate_day_result(
    score_date: pd.Timestamp,
    expected_cashdesk_names: List[str],
    day_result_df: pd.DataFrame,
) -> None:
    score_date = pd.Timestamp(score_date).normalize()
    if list(day_result_df.columns) != CHECKPOINT_COLUMNS:
        raise ValueError("Checkpoint имеет неверную схему")
    if day_result_df.empty:
        raise ValueError("Пустой checkpoint")
    if not pd.to_datetime(day_result_df["score_date"]).eq(score_date).all():
        raise ValueError("Checkpoint содержит другую дату скоринга")
    expected_report_date = score_date - pd.Timedelta(days=DATA_LAG_DAYS)
    if not pd.to_datetime(day_result_df["report_date"]).eq(expected_report_date).all():
        raise ValueError("Checkpoint рассчитан с другим T-2")
    checkpoint_cashdesks = set(
        day_result_df["atdtmco_cashdesk_name"].dropna().unique()
    )
    if checkpoint_cashdesks != set(expected_cashdesk_names):
        raise ValueError("Checkpoint содержит другой динамический набор касс")
    key_columns = ["score_date", "atdtmco_cashdesk_name", "forecast_date"]
    if day_result_df.duplicated(key_columns).any():
        raise ValueError("Checkpoint содержит дубли ключей")
    horizons = day_result_df.groupby("atdtmco_cashdesk_name")[
        "forecast_date"
    ].nunique()
    forecast_dates = pd.to_datetime(day_result_df["forecast_date"])
    expected_last_date = score_date + pd.Timedelta(days=FORECAST_DAYS - 1)
    if (
        len(day_result_df) != len(expected_cashdesk_names) * FORECAST_DAYS
        or not horizons.eq(FORECAST_DAYS).all()
        or forecast_dates.min() != score_date
        or forecast_dates.max() != expected_last_date
    ):
        raise ValueError("Checkpoint имеет неверный горизонт")
    if not np.isfinite(
        day_result_df[PREDICTION_COLUMNS].to_numpy(dtype=float)
    ).all():
        raise ValueError("Checkpoint содержит нечисловые central/q значения")
    flag_columns = ["fallback saldo", "fallback NS", "bootstrap NS"]
    if not all(
        pd.api.types.is_bool_dtype(day_result_df[column_name])
        for column_name in flag_columns
    ):
        raise ValueError("Флаги checkpoint должны иметь bool dtype")
    for fallback_column in ["fallback saldo", "fallback NS"]:
        if day_result_df.groupby("atdtmco_cashdesk_name")[
            fallback_column
        ].nunique().gt(1).any():
            raise ValueError("{} различается внутри задачи".format(fallback_column))
    if (day_result_df["fallback NS"] & day_result_df["bootstrap NS"]).any():
        raise ValueError("Fallback NS не может использовать bootstrap")


CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
retro_result_parts = []
with parallel_backend("loky", inner_max_num_threads=1):
    with Parallel(n_jobs=N_JOBS) as parallel:
        for score_index, score_date in enumerate(score_dates, start=1):
            score_date = pd.Timestamp(score_date).normalize()
            date_started_at = time.perf_counter()
            report_date, tasks = build_scoring_tasks(score_date, daily_df)
            expected_cashdesk_names = [task[1] for task in tasks]
            if not tasks:
                raise RuntimeError("Нет активных касс для {}".format(score_date.date()))
            checkpoint_path = CHECKPOINT_DIR / "closed_nan_{:%Y-%m-%d}.parquet".format(
                score_date
            )
            checkpoint_is_valid = False
            if USE_CHECKPOINTS and checkpoint_path.exists():
                try:
                    day_result_df = pd.read_parquet(checkpoint_path)
                    validate_day_result(
                        score_date,
                        expected_cashdesk_names,
                        day_result_df,
                    )
                    checkpoint_is_valid = True
                    status = "загружено"
                except Exception as checkpoint_error:
                    print("{} пересчитывается: {}".format(
                        score_date.date(), checkpoint_error
                    ))
            if not checkpoint_is_valid:
                day_result_parts = parallel(
                    delayed(forecast_cashdesk_task)(
                        score_date,
                        report_date,
                        cashdesk_index,
                        cashdesk_name,
                        cashdesk_df,
                    )
                    for cashdesk_index, cashdesk_name, cashdesk_df in tasks
                )
                day_result_df = pd.concat(day_result_parts, ignore_index=True)[
                    CHECKPOINT_COLUMNS
                ]
                validate_day_result(
                    score_date,
                    expected_cashdesk_names,
                    day_result_df,
                )
                temporary_path = checkpoint_path.with_suffix(".tmp.parquet")
                day_result_df.to_parquet(temporary_path, index=False)
                temporary_path.replace(checkpoint_path)
                status = "рассчитано"
            retro_result_parts.append(day_result_df)
            elapsed_seconds = time.perf_counter() - date_started_at
            print("[{}/{}] {}: {} — {:.1f} сек".format(
                score_index,
                len(score_dates),
                status,
                score_date.date(),
                elapsed_seconds,
            ))

retro_result_df = pd.concat(retro_result_parts, ignore_index=True)
actual_df = daily_df[[
    "atdtmco_cashdesk_name",
    "atdtmco_cashdesk_name_trn",
    "calday",
    "atdtmco_saldo_turn_fact",
    "atdtmco_ns_daily_min_raw",
    "atdtmco_ns_fact",
]].rename(columns={"calday": "forecast_date"})
retro_result_df = retro_result_df.merge(
    actual_df,
    on=["atdtmco_cashdesk_name", "forecast_date"],
    how="left",
    suffixes=("", "_actual"),
    validate="many_to_one",
)
retro_result_df["atdtmco_cashdesk_name_trn"] = retro_result_df[
    "atdtmco_cashdesk_name_trn_actual"
].combine_first(retro_result_df["atdtmco_cashdesk_name_trn"])
retro_result_df = retro_result_df.drop(
    columns="atdtmco_cashdesk_name_trn_actual"
)
open_schedule_df = (
    daily_df[["atdtmco_cashdesk_name", "calday"]]
    .drop_duplicates()
    .rename(columns={"calday": "forecast_date"})
    .assign(_source_daily_row_exists=True)
)
retro_result_df = retro_result_df.merge(
    open_schedule_df,
    on=["atdtmco_cashdesk_name", "forecast_date"],
    how="left",
    validate="many_to_one",
)
retro_result_df["касса закрыта"] = retro_result_df[
    "_source_daily_row_exists"
].isna()
retro_result_df.loc[
    retro_result_df["касса закрыта"],
    PREDICTION_COLUMNS,
] = 0.0
retro_result_df.loc[
    retro_result_df["касса закрыта"],
    "bootstrap NS",
] = False
retro_result_df = retro_result_df.drop(columns="_source_daily_row_exists")
print("Итог: {:,} строк, {:,} касс".format(
    len(retro_result_df),
    retro_result_df["atdtmco_cashdesk_name"].nunique(),
))

## 5. Общая first-day выборка и отчётные метрики

Варианты NS сравниваются на одной inner-joined выборке первого дня горизонта, где одновременно есть старый прогноз и отрицательный факт NS. Нулевые факты NS полностью исключены из ошибок и знаменателя невыдач. Невыдача определяется как `факт < прогноз`; `MAE95` удаляет `floor(5%)` наиболее отрицательных фактов.

Метрики `saldo_turn` также считаются по первому дню, но по всем валидным парам факт-прогноз, включая наблюдаемые нули. Для `MAE95` saldo исключаются `floor(5%)` фактов с наибольшим абсолютным значением; сравнение со старой saldo-моделью не выполняется.

In [ ]:
def mae95_ns(
    evaluation_df: pd.DataFrame,
    fact_col: str,
    pred_col: str,
) -> float:
    absolute_error = (evaluation_df[fact_col] - evaluation_df[pred_col]).abs()
    trim_count = int(np.floor(len(evaluation_df) * 0.05))
    excluded_indexes = evaluation_df[fact_col].nsmallest(trim_count).index
    return float(absolute_error.drop(index=excluded_indexes).mean())


def build_overall_metric_row(
    evaluation_df: pd.DataFrame,
    variant: str,
    quantile_label: str,
    pred_col: str,
    old_breach_rate: float,
) -> Dict[str, object]:
    error = evaluation_df["atdtmco_ns_fact"] - evaluation_df[pred_col]
    absolute_error = error.abs()
    breach = evaluation_df["atdtmco_ns_fact"] < evaluation_df[pred_col]
    breach_rate = float(breach.mean())
    return {
        "вариант": variant,
        "квантиль": quantile_label,
        "количество строк": len(evaluation_df),
        "количество касс": evaluation_df["atdtmco_cashdesk_name"].nunique(),
        "количество невыдач": int(breach.sum()),
        "% невыдач": breach_rate,
        "отклонение от старого % невыдач": breach_rate - old_breach_rate,
        "MAE": absolute_error.mean(),
        "MAE95": mae95_ns(evaluation_df, "atdtmco_ns_fact", pred_col),
        "медианная абсолютная ошибка": absolute_error.median(),
        "P90 абсолютной ошибки": absolute_error.quantile(0.90),
        "смещение факт − прогноз": error.mean(),
    }


first_day_df = retro_result_df[
    retro_result_df["forecast_date"].eq(retro_result_df["score_date"])
].copy()
common_first_day_df = first_day_df.merge(
    old_history_dedup_df[["cashdesk_name", "calday", "atdtmco_ns_pred_old"]],
    left_on=["atdtmco_cashdesk_name_trn", "forecast_date"],
    right_on=["cashdesk_name", "calday"],
    how="inner",
    validate="many_to_one",
).dropna(subset=["atdtmco_ns_fact", "atdtmco_ns_pred_old"])
common_first_day_df = common_first_day_df[
    common_first_day_df["atdtmco_ns_fact"] < 0
].reset_index(drop=True)
if common_first_day_df.empty:
    raise RuntimeError("Нет общей first-day выборки с отрицательным фактом и old")

old_breach_rate = float((
    common_first_day_df["atdtmco_ns_fact"]
    < common_first_day_df["atdtmco_ns_pred_old"]
).mean())
overall_rows = [
    build_overall_metric_row(
        common_first_day_df,
        "старый forecast_model",
        "—",
        "atdtmco_ns_pred_old",
        old_breach_rate,
    ),
    build_overall_metric_row(
        common_first_day_df,
        "центральный SARIMA без ошибок",
        "—",
        "atdtmco_ns_pred_central",
        old_breach_rate,
    ),
]
for quantile in BOOTSTRAP_QUANTILES:
    quantile_label = "q{:02d}".format(int(round(100 * float(quantile))))
    overall_rows.append(build_overall_metric_row(
        common_first_day_df,
        quantile_label,
        quantile_label,
        QUANTILE_COLUMNS[float(quantile)],
        old_breach_rate,
    ))
overall_model_comparison_df = pd.DataFrame(overall_rows)


def build_cashdesk_q01_row(cashdesk_df: pd.DataFrame) -> Dict[str, object]:
    fact_col = "atdtmco_ns_fact"
    old_col = "atdtmco_ns_pred_old"
    central_col = "atdtmco_ns_pred_central"
    q01_col = QUANTILE_COLUMNS[0.01]
    old_breach = cashdesk_df[fact_col] < cashdesk_df[old_col]
    central_breach = cashdesk_df[fact_col] < cashdesk_df[central_col]
    q01_breach = cashdesk_df[fact_col] < cashdesk_df[q01_col]
    return {
        "касса": cashdesk_df["atdtmco_cashdesk_name"].iloc[0],
        "количество строк": len(cashdesk_df),
        "старый: количество невыдач": int(old_breach.sum()),
        "старый: % невыдач": old_breach.mean(),
        "центральный: количество невыдач": int(central_breach.sum()),
        "центральный: % невыдач": central_breach.mean(),
        "q01: количество невыдач": int(q01_breach.sum()),
        "q01: % невыдач": q01_breach.mean(),
        "q01 минус старый, п.п.": 100.0 * (q01_breach.mean() - old_breach.mean()),
        "MAE95 старый": mae95_ns(cashdesk_df, fact_col, old_col),
        "MAE95 центральный": mae95_ns(cashdesk_df, fact_col, central_col),
        "MAE95 q01": mae95_ns(cashdesk_df, fact_col, q01_col),
        "смещение факт − q01": (cashdesk_df[fact_col] - cashdesk_df[q01_col]).mean(),
    }


q01_by_cashdesk_df = pd.DataFrame([
    build_cashdesk_q01_row(cashdesk_df)
    for _, cashdesk_df in common_first_day_df.groupby(
        "atdtmco_cashdesk_name",
        sort=False,
    )
]).sort_values(
    ["q01: % невыдач", "касса"],
    ascending=[False, True],
    kind="mergesort",
).reset_index(drop=True)

method_events_df = (
    retro_result_df
    .groupby(["score_date", "atdtmco_cashdesk_name"], as_index=False)
    .agg(**{
        "fallback saldo": ("fallback saldo", "first"),
        "fallback NS": ("fallback NS", "first"),
        "открытых шагов": ("касса закрыта", lambda values: int((~values).sum())),
        "открытых шагов с bootstrap": ("bootstrap NS", "sum"),
    })
)
method_events_df["cashdesk-date с bootstrap"] = (
    method_events_df["открытых шагов с bootstrap"] > 0
)


def build_bootstrap_coverage_row(
    method_df: pd.DataFrame,
    cashdesk_name: str,
) -> Dict[str, object]:
    cashdesk_dates = len(method_df)
    open_steps = int(method_df["открытых шагов"].sum())
    fallback_ns_count = int(method_df["fallback NS"].sum())
    fallback_saldo_count = int(method_df["fallback saldo"].sum())
    any_bootstrap_count = int(method_df["cashdesk-date с bootstrap"].sum())
    bootstrap_open_steps = int(method_df["открытых шагов с bootstrap"].sum())
    return {
        "касса": cashdesk_name,
        "количество касса-дней": cashdesk_dates,
        "касса-дней с резервным прогнозом NS": fallback_ns_count,
        "доля касса-дней с резервным прогнозом NS": (
            fallback_ns_count / cashdesk_dates
        ),
        "касса-дней с резервным прогнозом saldo": fallback_saldo_count,
        "доля касса-дней с резервным прогнозом saldo": (
            fallback_saldo_count / cashdesk_dates
        ),
        "касса-дней хотя бы с одним бутстрэп-шагом": any_bootstrap_count,
        "доля касса-дней хотя бы с одним бутстрэп-шагом": (
            any_bootstrap_count / cashdesk_dates
        ),
        "открытых шагов с бутстрэпом": bootstrap_open_steps,
        "доля открытых шагов с бутстрэпом": (
            bootstrap_open_steps / open_steps if open_steps else np.nan
        ),
    }


bootstrap_coverage_rows = [build_bootstrap_coverage_row(method_events_df, "ВСЕ")]
for cashdesk_name, cashdesk_method_df in method_events_df.groupby(
    "atdtmco_cashdesk_name",
    sort=True,
):
    bootstrap_coverage_rows.append(build_bootstrap_coverage_row(
        cashdesk_method_df,
        cashdesk_name,
    ))
bootstrap_coverage_df = pd.DataFrame(bootstrap_coverage_rows)


def median_smoothness_ratio(
    evaluation_df: pd.DataFrame,
    fact_col: str,
    pred_col: str,
) -> float:
    ratios = []
    for _, group_df in evaluation_df.groupby("atdtmco_cashdesk_name"):
        ordered_df = group_df.sort_values("forecast_date")
        consecutive_mask = ordered_df["forecast_date"].diff().dt.days.eq(1)
        fact_variation = ordered_df[fact_col].diff().abs()[consecutive_mask].sum()
        pred_variation = ordered_df[pred_col].diff().abs()[consecutive_mask].sum()
        if fact_variation > 0:
            ratios.append(pred_variation / fact_variation)
    return float(np.median(ratios)) if ratios else np.nan


def build_saldo_metric_row(
    evaluation_df: pd.DataFrame,
    calculation_level: str,
    cashdesk_name: str,
) -> Dict[str, object]:
    fact_col = "atdtmco_saldo_turn_fact"
    pred_col = "atdtmco_saldo_turn_pred"
    clean_df = evaluation_df.dropna(subset=[fact_col, pred_col]).copy()
    error = clean_df[fact_col] - clean_df[pred_col]
    absolute_error = error.abs()
    trim_count = int(np.floor(len(clean_df) * 0.05))
    excluded_indexes = clean_df[fact_col].abs().nlargest(trim_count).index
    denominator = clean_df[fact_col].abs().sum()
    return {
        "показатель": "saldo_turn",
        "модель": "центральный guarded SARIMA",
        "уровень расчёта": calculation_level,
        "касса": cashdesk_name,
        "количество строк": len(clean_df),
        "количество касс": clean_df["atdtmco_cashdesk_name"].nunique(),
        "количество дат": clean_df["forecast_date"].nunique(),
        "средний факт": clean_df[fact_col].mean(),
        "средний прогноз": clean_df[pred_col].mean(),
        "MAE": absolute_error.mean(),
        "MAE95": absolute_error.drop(index=excluded_indexes).mean(),
        "медианная абсолютная ошибка": absolute_error.median(),
        "P90 абсолютной ошибки": absolute_error.quantile(0.90),
        "смещение факт − прогноз": error.mean(),
        "WAPE": absolute_error.sum() / denominator if denominator > 0 else np.nan,
        "медианный коэффициент сглаженности": median_smoothness_ratio(
            clean_df,
            fact_col,
            pred_col,
        ),
    }


saldo_first_day_df = first_day_df.dropna(subset=[
    "atdtmco_saldo_turn_fact",
    "atdtmco_saldo_turn_pred",
]).copy()
if saldo_first_day_df.empty:
    raise RuntimeError("Нет first-day строк для метрик saldo")
saldo_metric_rows = [build_saldo_metric_row(
    saldo_first_day_df,
    "в целом",
    "ВСЕ",
)]
for cashdesk_name, cashdesk_saldo_df in saldo_first_day_df.groupby(
    "atdtmco_cashdesk_name",
    sort=True,
):
    saldo_metric_rows.append(build_saldo_metric_row(
        cashdesk_saldo_df,
        "по кассе",
        cashdesk_name,
    ))
saldo_metrics_summary_df = pd.DataFrame(saldo_metric_rows)

metric_formats = {
    "% невыдач": "{:.2%}",
    "отклонение от старого % невыдач": "{:+.2%}",
    "MAE": "{:,.0f}",
    "MAE95": "{:,.0f}",
    "медианная абсолютная ошибка": "{:,.0f}",
    "P90 абсолютной ошибки": "{:,.0f}",
    "смещение факт − прогноз": "{:,.0f}",
}
print("Старый целевой % невыдач: {:.2%}".format(old_breach_rate))
display(overall_model_comparison_df.style.format(metric_formats))
display(q01_by_cashdesk_df.style.format({
    "старый: % невыдач": "{:.2%}",
    "центральный: % невыдач": "{:.2%}",
    "q01: % невыдач": "{:.2%}",
    "q01 минус старый, п.п.": "{:+.2f}",
    "MAE95 старый": "{:,.0f}",
    "MAE95 центральный": "{:,.0f}",
    "MAE95 q01": "{:,.0f}",
    "смещение факт − q01": "{:,.0f}",
}))
display(bootstrap_coverage_df.style.format({
    "доля касса-дней с резервным прогнозом NS": "{:.2%}",
    "доля касса-дней с резервным прогнозом saldo": "{:.2%}",
    "доля касса-дней хотя бы с одним бутстрэп-шагом": "{:.2%}",
    "доля открытых шагов с бутстрэпом": "{:.2%}",
}))
display(saldo_metrics_summary_df.style.format({
    "средний факт": "{:,.0f}",
    "средний прогноз": "{:,.0f}",
    "MAE": "{:,.0f}",
    "MAE95": "{:,.0f}",
    "медианная абсолютная ошибка": "{:,.0f}",
    "P90 абсолютной ошибки": "{:,.0f}",
    "смещение факт − прогноз": "{:,.0f}",
    "WAPE": "{:.2%}",
    "медианный коэффициент сглаженности": "{:.2f}",
}, na_rep="—"))

## Как читать варианты руководителю

- **Центральный SARIMA без ошибок** — ожидаемая траектория NS без поправки на исторические ошибки, физически ограниченная сверху нулём. Если модель кассы не проходит защитные проверки, используется устойчивый прогноз по дням недели.
- **q01..q15** — односторонние консервативные варианты: к центральной траектории по каждому шагу добавляется распределение исторических ошибок скользящей ретро-проверки. Если ошибок для отдельного шага недостаточно, только этот шаг получает консервативную поправку 15%.
- Все q-варианты заданы единой глобальной сеткой и **не калибруются отдельно по кассам**.
- **Центральный saldo_turn** — отдельный guarded SARIMA без bootstrap; при отказе используется прогноз по дням недели.
- На будущих датах без строки в исходном `daily_df` прогнозы NS и saldo обнуляются как для закрытой кассы; фактический ноль сам по себе закрытием не считается.
- Ошибки и доли невыдач NS рассчитаны на общей выборке первого дня со старой моделью и только при `факт NS < 0`. Метрики saldo используют все валидные факты первого дня, включая нули.